In [2]:

import os
import dill
import spacy
import pickle




# Import all necessary items from the olaf package
from olaf import Pipeline
from olaf.pipeline.pipeline_component.term_extraction import (
    POSTermExtraction,
    TFIDFTermExtraction,
    ManualCandidateTermExtraction
    )
from olaf.pipeline.pipeline_component.concept_relation_extraction import (
    CTsToConceptExtraction,
    CTsToRelationExtraction,
    SynonymRelationExtraction,
    SynonymConceptExtraction,
    AgglomerativeClusteringRelationExtraction,
    AgglomerativeClusteringConceptExtraction,
    LLMBasedConceptExtraction,
    LLMBasedRelationExtraction
)
from olaf.pipeline.pipeline_component.axiom_extraction.owl_axiom_extraction import OWLAxiomExtraction
from olaf.data_container.knowledge_representation_schema import KnowledgeRepresentation
from olaf.repository.serialiser import BaseOWLSerialiser
from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader
from olaf.data_container import CandidateTerm, Relation, Concept


/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 11.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [3]:
# Load the spacy language model according to the corpus
spacy_model = spacy.load("en_core_web_lg")

In [4]:
# Initialise the corpus (for this example text version)
corpus = TextCorpusLoader(
    corpus_path="../data/metal/Casting_defect.txt",
)

In [6]:
bad_concept_pos = ["VERB","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]
bad_relation_pos = ["NOUN","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]

def candidates_post_processing(candidates: set[CandidateTerm], bad_pos ) -> set[CandidateTerm]:
	
	list_label = []
	new_candidates = set()
	for candidate in candidates:
		keep = True
		if len(candidate.corpus_occurrences) > 0:
			for co in candidate.corpus_occurrences:
				for token in co:
					if (token.is_punct or token.is_stop or token.pos in bad_pos):
						keep = False
						break
			# print("good candidate: ", candidate.label)
		else:
			keep = False
			
		if keep and candidate.label not in list_label:
			new_candidates.add(candidate)
			list_label.append(candidate.label)
	return new_candidates

concept_post_processing = lambda candidates: candidates_post_processing(candidates, bad_concept_pos)
relation_post_processing = lambda candidates: candidates_post_processing(candidates, bad_relation_pos)

In [7]:
def clean_relations(kr: KnowledgeRepresentation):
    """
    Clean the relations in the knowledge representation
    :param kr: KnowledgeRepresentation
    :return: None
    """
    relations_to_remove = []
    for relation in kr.relations:
        if relation.source_concept is None or relation.destination_concept is None:
            relations_to_remove.append(relation)
        elif relation.source_concept.label == relation.destination_concept.label:
            relations_to_remove.append(relation)
    for relation in relations_to_remove:
        kr.relations.remove(relation)

def serialize_pipeline(pipeline, file_path):
    """
    Serialize the pipeline to a file
    :param pipeline: Pipeline
    :param file_path: str
    :return: None
    """
    components = pipeline.pipeline_components
    with open(file_path, 'wb') as f:
        dill.dump(components, f)

def deserialize_pipeline(file_path):
    """
    Deserialize the pipeline from a file
    :param file_path: str
    :return: Pipeline
    """
    with open(file_path, 'rb') as f:
        components = dill.load(f)
    return Pipeline(
        spacy_model=spacy_model,
        pipeline_components=components,
    )


# Pipeline 1
     - POSTermExtraction
     - CTsToConceptExtraction
     - POSTermExtraction
     - CTsToRelationExtraction


In [ ]:
import itertools


post_term_concept_components = [
    POSTermExtraction(pos_selection=["NOUN"])
]

ct_to_concept_components = [
    CTsToConceptExtraction()
]

post_term_relation_components = [
    POSTermExtraction(pos_selection=["VERB"])
]

ct_to_relation_components = [
    CTsToConceptExtraction()
]


pipelines_components = list(itertools.product(
	post_term_concept_components,
	ct_to_concept_components,
	post_term_relation_components,
	ct_to_relation_components
))

pipelines = [
    Pipeline(
		spacy_model=spacy_model,
		pipeline_components=pipeline_components,
        corpus_loader=corpus,
	)
	for pipeline_components in pipelines_components
]



[2025-05-13 06:40:28,814] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 06:40:28,815] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2025-05-13 06:40:28,815] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 06:40:28,816] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


POSTermExtraction
CTsToConceptExtraction
POSTermExtraction
CTsToConceptExtraction
